In [10]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer
import os

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from torch import nn
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm

In [11]:
TRAIN = pd.read_csv("data/train.csv")
TEST = pd.read_csv("data/test.csv")

TRAIN.info()
TEST.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52889 entries, 0 to 52888
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                52889 non-null  object
 1   sumber            52889 non-null  object
 2   kodekat           52889 non-null  object
 3   kodesubkat        52889 non-null  object
 4   kategori          52889 non-null  object
 5   subkategori       52889 non-null  object
 6   katakunci         52889 non-null  object
 7   tanggal           52889 non-null  object
 8   judul_berita      52889 non-null  object
 9   konten_berita     52889 non-null  object
 10  nama_tokoh        52889 non-null  object
 11  jabatan           52889 non-null  object
 12  organisasi        52889 non-null  object
 13  lokasi            52889 non-null  object
 14  alias             52889 non-null  object
 15  kutipan           52889 non-null  object
 16  sentimen_kutipan  52889 non-null  object
 17  sentimen_ber

In [12]:
PRETRAINED_FOLDER = "pretrained"   # UBAH
MODEL_NAME = "indobert"            # UBAH
TEXT = "konten_berita"                      # UBAH
TARGET = "sentimen_berita"                   # UBAH
MAX_LENGTH = 128                   # UBAH: lihat persentil di bawah
BATCH_SIZE = 16                    # UBAH
EPOCHS = 4                         # UBAH: 3–5 biasanya cukup
LEARNING_RATE = 3e-5               # UBAH: 1e-5 .. 5e-5
PATIENCE = 2                       # berhenti kalau F1 val tidak naik 2 epoch berturut-turut
SEED = 42                          # UBAH
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"        # UBAH: False kalau GPU lama / error bf16
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(f"{PRETRAINED_FOLDER}/{MODEL_NAME}")
TRAIN[TEXT] = TRAIN[TEXT].fillna("").astype(str)
TEST[TEXT] = TEST[TEXT].fillna("").astype(str)
token_counts = []
for text in TRAIN[TEXT].head(2000):
    token_counts.append(len(tokenizer(text)["input_ids"]))
print("token per teks, persentil 50/95/99:", np.percentile(token_counts, [50, 95, 99]))

token per teks, persentil 50/95/99: [ 462.5  1022.05 1666.11]


In [13]:
import re, string

# 1. bersihin teks dulu (di train DAN test, pakai fungsi yang sama)
def bersihkan(teks):
    teks = teks.lower()

    # hapus pola lokasi di awal kalimat, misalnya "jakarta - ", "jayapura - ", "australia - "
    teks = re.sub(r'^[a-zéèáàäâõóòöôç\s]+-\s+', '', teks, flags=re.IGNORECASE)

    # hapus bagian "baca juga:" atau "lihat juga:" beserta teks di antara \n\n sampai \n\n
    teks = re.sub(r'(baca juga:|lihat juga:).*?\n\n.*?\n\n', ' ', teks, flags=re.DOTALL | re.IGNORECASE)

    # hapus pola tanggal seperti (12/3/22), (1/12/2021)
    teks = re.sub(r'\(\d{1,2}/\d{1,2}/\d{2,4}\)', '', teks)

    # hapus string \n\n dan variasinya
    teks = re.sub(r'\\n+|\n+', ' ', teks)

    # hapus pola promosi halaman dan video khas detik.com
    teks = re.sub(r'selengkapnya ada di halaman berikutnya>>\s*selanjutnya', ' ', teks, flags=re.IGNORECASE)
    teks = re.sub(r'baca juga:.*?simak video.*?\[gambas:video.*?\]', ' ', teks, flags=re.DOTALL | re.IGNORECASE)
    teks = re.sub(r'\(fykfyk\)', '', teks)

    # hapus kata dalam kurung beserta kurungnya
    teks = re.sub(r'\([^)]*\)', ' ', teks)

    # hapus link/url (http, https, www)
    teks = re.sub(r'http\S+|www\.\S+', ' ', teks)

    # hapus baris atau kalimat yang diawali dengan pola wartawan/editor seperti:
    # "penulis:", "editor:", "reporter:", "oleh:", "pewarta:"
    teks = re.sub(r'\b(penulis|editor|reporter|oleh|pewarta)\s*[:\-]?\s*[a-z\s\.]+', ' ', teks, flags=re.IGNORECASE)

    teks = re.sub(r'(^|\n)\s*(penulis|editor|reporter|oleh|pewarta)\s*[:\-]\s*[^\n]+', ' ', teks, flags=re.IGNORECASE)

    # hapus nama hari (ID & EN)
    hari = r'\b(senin|selasa|rabu|kamis|jumat|jum\'at|sabtu|minggu|ahad|monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b'
    teks = re.sub(hari, ' ', teks, flags=re.IGNORECASE)

    # hapus nama bulan (ID & EN)
    bulan = r'\b(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\b'
    teks = re.sub(bulan, ' ', teks, flags=re.IGNORECASE)

    # hapus simbol hak cipta seperti "¬© antara", "©", "¬©", dsb
    teks = re.sub(r'¬?©\s*\w*', ' ', teks)
    teks = re.sub(r'\bantara\b', ' ', teks)  # hapus "antara" jika muncul sebagai watermark di akhir teks

    # hapus angka dan simbol tidak penting
    teks = re.sub(r'\d+', ' ', teks)  # hapus angka
    teks = re.sub(r'[“”‘’•–…•—•−·–—_=:;<>|~`^]', ' ', teks)  # hapus simbol umum
    teks = teks.translate(str.maketrans('', '', string.punctuation))  # hapus tanda baca standar

    # rapikan spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

TRAIN[TEXT] = TRAIN[TEXT].apply(bersihkan)
TEST[TEXT] = TEST[TEXT].apply(bersihkan)

In [14]:
from torch.utils.data import Dataset

CLASSES = sorted(TRAIN[TARGET].unique())
TRAIN["target"] = TRAIN[TARGET].map(dict(zip(CLASSES, range(len(CLASSES)))))
TEST["target"] = 0                                # pengisi, tidak dipakai


class TextDataset(Dataset):
    def __init__(self, table):
        self.tokens = tokenizer(table[TEXT].tolist(), truncation=True, max_length=MAX_LENGTH)
        self.targets = table["target"].tolist()

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        item = {"input_ids": self.tokens["input_ids"][i],
                "attention_mask": self.tokens["attention_mask"][i]}
        item["labels"] = self.targets[i]
        return item


print(TextDataset(TRAIN.head(2))[0])

{'input_ids': [2, 4408, 9281, 10380, 4318, 664, 126, 9648, 4049, 1147, 5397, 27519, 4476, 30356, 10276, 30370, 34, 234, 9921, 1578, 27843, 988, 393, 19, 3254, 2264, 14656, 501, 3921, 2077, 2906, 245, 22062, 79, 5731, 4522, 262, 6189, 2195, 9921, 3693, 339, 1456, 41, 8796, 9010, 464, 5053, 4198, 3921, 339, 8227, 11436, 4198, 776, 4522, 262, 661, 4887, 2120, 13796, 30357, 8687, 4152, 4235, 1697, 112, 19424, 34, 18248, 26, 4408, 19424, 256, 18248, 4887, 2120, 112, 5842, 13047, 7317, 26, 4486, 28769, 4408, 6810, 4887, 186, 11436, 4198, 10782, 1835, 600, 1633, 7147, 7513, 2011, 823, 23331, 41, 3574, 988, 6680, 1835, 600, 79, 2919, 1496, 119, 6039, 711, 985, 282, 823, 450, 7317, 4486, 2337, 2075, 885, 830, 370, 20571, 150, 24782, 41, 281, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [15]:
part_train, part_val = train_test_split(TRAIN, test_size=0.2, stratify=TRAIN["target"],
                                        random_state=42)
collate = DataCollatorWithPadding(tokenizer)
train_loader = DataLoader(TextDataset(part_train), batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate)
val_loader = DataLoader(TextDataset(part_val), batch_size=BATCH_SIZE * 2, collate_fn=collate)
test_loader = DataLoader(TextDataset(TEST), batch_size=BATCH_SIZE * 2, collate_fn=collate)
print(len(CLASSES), "kelas |", len(part_train), "train |", len(part_val), "val")

3 kelas | 42311 train | 10578 val


In [16]:
network = AutoModelForSequenceClassification.from_pretrained(
    f"{PRETRAINED_FOLDER}/{MODEL_NAME}", num_labels=len(CLASSES),
    id2label=dict(zip(range(len(CLASSES)), CLASSES)))
network = network.to(DEVICE)
loss_function = nn.CrossEntropyLoss()
# counts = np.bincount(part_train["target"], minlength=len(CLASSES))          # kelas timpang
# class_weight = torch.tensor(len(part_train) / (len(CLASSES) * counts), dtype=torch.float)
# loss_function = nn.CrossEntropyLoss(weight=class_weight.to(DEVICE))
optimizer = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
print("langkah total:", total_steps)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: pretrained/indobert
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


langkah total: 10580


In [17]:
import numpy as np
import torch
from tqdm import tqdm


def run_epoch(loader, training):
    network.train(training)
    total_loss, probabilities = 0.0, []
    for batch in tqdm(loader, leave=False):
        batch = batch.to(DEVICE)
        targets = batch.pop("labels")
        with torch.set_grad_enabled(training):
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                logits = network(**batch).logits.float()
            loss = loss_function(logits, targets)
        if training:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        total_loss += loss.item() * len(targets)
        probability = torch.softmax(logits.detach(), dim=1)
        probabilities.append(probability.cpu().numpy())
    return total_loss / len(loader.dataset), np.concatenate(probabilities)


val_loss, val_probabilities = run_epoch(val_loader, training=False)
print("sebelum latih | val loss:", round(val_loss, 3), "| bentuk:", val_probabilities.shape)

sebelum latih | val loss: 1.223 | bentuk: (10578, 3)


In [18]:
from sklearn.metrics import f1_score

best_f1, waited = -1.0, 0
for epoch in range(EPOCHS):
    train_loss, _ = run_epoch(train_loader, training=True)
    val_loss, val_probabilities = run_epoch(val_loader, training=False)
    val_f1 = f1_score(part_val["target"], val_probabilities.argmax(axis=1), average="macro")
    print(f"epoch {epoch + 1}: train {train_loss:.3f} | val {val_loss:.3f} | F1 {val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1, waited = val_f1, 0
        torch.save(network.state_dict(), "outputs/best_model.pt")
    else:
        waited += 1
    if waited >= PATIENCE:
        print("berhenti dini")
        break
print("F1 val terbaik:", round(best_f1, 4))

epoch 1: train 0.850 | val 0.759 | F1 0.5795


epoch 2: train 0.667 | val 0.699 | F1 0.6581


epoch 3: train 0.463 | val 0.739 | F1 0.6795


epoch 4: train 0.282 | val 0.950 | F1 0.6801
F1 val terbaik: 0.6801


In [19]:
import torch
from sklearn.metrics import classification_report

network.load_state_dict(torch.load("outputs/best_model.pt"))
_, val_probabilities = run_epoch(val_loader, training=False)
part_val = part_val.copy()
part_val["pred"] = np.array(CLASSES)[val_probabilities.argmax(axis=1)]
print(classification_report(part_val[TARGET], part_val["pred"], digits=4))
wrong = part_val[part_val[TARGET] != part_val["pred"]]
print(wrong[[TEXT, TARGET, "pred"]].head(6).to_string(max_colwidth=70))

              precision    recall  f1-score   support

     Negatif     0.6603    0.5850    0.6203      1648
      Netral     0.6062    0.6234    0.6147      3396
     Positif     0.7988    0.8121    0.8054      5534

    accuracy                         0.7161     10578
   macro avg     0.6884    0.6735    0.6801     10578
weighted avg     0.7154    0.7161    0.7153     10578

                                                               konten_berita sentimen_berita     pred
15305  jakarta terdapat beberapa berita humaniora yang menarik perhatian ...          Netral  Positif
31977  jakarta teknologi automasi dapat membantu meningkatkan produktivit...         Positif   Netral
30821  penularan covid trennya turun angka kesembuhan terus naik angka ke...         Positif   Netral
21903  jakarta anggota badan legislasi dpr ri darori wonodipuro mengingin...          Netral  Positif
455    jakarta plt juru bicara kpk ali fikri menyatakan pihaknya akan men...         Negatif   Netral
46447  

In [20]:
import numpy as np
import pandas as pd

_, test_probabilities = run_epoch(test_loader, training=False)
proba_table = pd.DataFrame(test_probabilities, columns=CLASSES)
proba_table.insert(0, "id", TEST["id"].to_numpy())
proba_table.to_csv(f"outputs/test_proba_seed{SEED}.csv", index=False)

submission = pd.DataFrame()
submission["id"] = TEST["id"]                                               # UBAH
submission["label"] = np.array(CLASSES)[test_probabilities.argmax(axis=1)]  # UBAH
submission.to_csv("submission.csv", index=False)
print(submission["label"].value_counts())

label
Positif    12129
Netral      7397
Negatif     3139
Name: count, dtype: int64
